<a href="https://www.kaggle.com/code/avikdas567/bm25-tf-idf-reranking-legal-information-retrieval?scriptVersionId=307425732" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import gc
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Load Data

BASE_PATH = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval"

laws = pd.read_csv(f"{BASE_PATH}/laws_de.csv")
court = pd.read_csv(f"{BASE_PATH}/court_considerations.csv")
test = pd.read_csv(f"{BASE_PATH}/test.csv")

corpus = pd.concat([laws, court], ignore_index=True).dropna()

texts = corpus["text"].astype(str).tolist()
citations = corpus["citation"].tolist()

print(f"Corpus size: {len(texts)}")

# Tokenization

import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9äöüß]", " ", text)
    return text.split()

tokenized_corpus = [tokenize(t) for t in tqdm(texts, desc="Tokenizing")]

# BM25

from collections import defaultdict, Counter
import math

N = len(tokenized_corpus)
doc_len = np.array([len(doc) for doc in tokenized_corpus])
avgdl = doc_len.mean()

inv_index = defaultdict(list)

for i, doc in enumerate(tokenized_corpus):
    freq = Counter(doc)
    for term, f in freq.items():
        inv_index[term].append((i, f))

idf = {t: math.log((N - len(p) + 0.5)/(len(p)+0.5)+1) for t,p in inv_index.items()}

k1, b = 1.5, 0.75

def bm25_score(query_tokens):
    scores = defaultdict(float)
    for term in query_tokens:
        if term not in inv_index:
            continue
        for doc_id, f in inv_index[term]:
            denom = f + k1*(1-b+b*doc_len[doc_id]/avgdl)
            scores[doc_id] += idf[term]*(f*(k1+1)/denom)
    return scores

# TF-IDF (Embedding Cache)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=80000, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(texts)

# Query Expansion

def expand_query(q):
    q = q.lower()
    expansions = []

    mapping = {
        "contract": ["vertrag", "schuld", "pflicht"],
        "damage": ["schaden", "haftung"],
        "liability": ["haftung"],
        "employment": ["arbeit", "arbeitsvertrag"],
        "property": ["eigentum", "besitz"],
        "insurance": ["versicherung"],
        "termination": ["kündigung"],
        "company": ["gesellschaft"],
        "lease": ["miete"],
        "loan": ["darlehen"],
        "law": ["gesetz"],
        "court": ["gericht"],
        "claim": ["anspruch"],
        "right": ["recht"]
    }

    for key, vals in mapping.items():
        if key in q:
            expansions += vals

    return q + " " + " ".join(expansions)

# Reranker

def rerank(query, candidates, bm25_scores, tfidf_scores):
    q_tokens = set(tokenize(query))

    bm25_max = max(bm25_scores.values()) if bm25_scores else 1e-6
    tfidf_max = tfidf_scores.max() if len(tfidf_scores) > 0 else 1e-6

    scored = []

    for idx in candidates:
        doc_tokens = set(tokenized_corpus[idx])

        overlap = len(q_tokens & doc_tokens) / (len(q_tokens)+1)
        length_norm = 1 / (1 + abs(len(doc_tokens) - 100))

        density = len(doc_tokens) / (len(tokenized_corpus[idx]) + 1)

        citation = citations[idx].lower()
        type_bonus = 0
        if "art." in citation:
            type_bonus += 0.05
        if "bge" in citation:
            type_bonus += 0.05

        score = (
            0.45 * (bm25_scores.get(idx, 0) / bm25_max) +
            0.35 * (tfidf_scores[idx] / tfidf_max) +
            0.15 * overlap +
            0.03 * length_norm +
            0.02 * density +
            type_bonus
        )

        scored.append((idx, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [i for i,_ in scored]

# Retrieval

def retrieve(query, top_k=120):
    query_exp = expand_query(query)
    tokens = tokenize(query_exp)

    bm25_scores = bm25_score(tokens)
    bm25_top = sorted(bm25_scores.items(), key=lambda x:x[1], reverse=True)[:top_k]
    bm25_idx = [i for i,_ in bm25_top]

    q_vec = vectorizer.transform([query_exp])
    tfidf_scores = (tfidf_matrix @ q_vec.T).toarray().ravel()

    tfidf_idx = np.argpartition(tfidf_scores, -top_k)[-top_k:]

    candidates = list(set(bm25_idx) | set(tfidf_idx))

    reranked = rerank(query_exp, candidates, bm25_scores, tfidf_scores)

    return reranked[:top_k]

# Dynamic Selection

def select_citations(indices):
    selected = []

    for i, idx in enumerate(indices):
        if i < 3:
            selected.append(citations[idx])
        elif i < 12:
            if i % 2 == 0:
                selected.append(citations[idx])
        elif i < 25:
            if i % 3 == 0:
                selected.append(citations[idx])
        else:
            break

    return list(dict.fromkeys(selected))

# Inference

print("Running inference...")
start = time.time()

predictions = []

for i, q in enumerate(tqdm(test["query"], desc="Inference")):
    idxs = retrieve(q)
    preds = select_citations(idxs)
    pred_str = ";".join(preds)
    predictions.append(pred_str)

    if i < 5:
        print("\n======================")
        print(f"Query {i+1}:")
        print(q[:200])
        print("\nTop Predictions:")
        print(pred_str[:300])
        print("======================")

print(f"Inference time: {time.time()-start:.2f}s")

# Submission

submission = pd.DataFrame({
    "query_id": test["query_id"],
    "predicted_citations": predictions
})

submission.to_csv("submission.csv", index=False)

print("\n'submission.csv' created.")

Corpus size: 175933


Tokenizing:   0%|          | 0/175933 [00:00<?, ?it/s]

Running inference...


Inference:   0%|          | 0/40 [00:00<?, ?it/s]


Query 1:
Four U.S.-based software companies (NorthWave Inc., Orion Systems LLC, ClearPeak Corp. and HarborSoft Ltd.) assert that R. Silva, who until April 2019 worked in the Milan development unit of Orion Sys

Top Predictions:
Art. 197e Abs. 2 AVO;Art. 43 Abs. 4 VNF;Art. 3 632.324.27;Art. 2 Abs. 2 956.126;Art. 2 Abs. 1 956.124;Art. 8 Abs. 1 BetmPV;Art. 2 830.22;Art. 3 Abs. 1 221.434;Art. 64 RelV-FINMA;Art. 25 Abs. 6 RelV-FINMA;Art. 4 414.132.2;Art. 4 VUKBK

Query 2:
On 9 August 2011 a 62‑year‑old cyclist (the claimant) was struck at a junction inside an industrial estate by a delivery van that had been leaving a side access to turn left onto a county road; the cy

Top Predictions:
Art. 197e Abs. 2 AVO;Art. 3 Abs. 1 221.434;Art. 43 Abs. 4 VNF;Art. 43 Abs. 3 VNF;Art. 92 Abs. 1 ERV;Art. 2 Abs. 2 956.126;Art. 2 Abs. 1 956.124;Art. 2 830.22;Art. 3 Abs. 2 221.434;Art. 33 Abs. 1 BankV;Art. 60 Abs. 1 BGerR;Art. 24 Abs. 1 VRV

Query 3:
On 12 March 2012, Meridian Leasing Ltd and Orion Transpor